# Visualize Best-Checkpoint Live Vehicle Replay

Use this notebook to restore one or more RLlib runs, auto-select a retained best-validation checkpoint for each run, replay a TraCI evaluation episode, and render the live vehicle movement as GIFs.

## What this notebook does

1. Locates the repo root and Python environment.
2. Points at one or more Hydra RLlib `RUN_DIRS`.
3. Restores the best retained validation checkpoint for each run.
4. Replays one evaluation episode on TraCI for each run.
5. Saves `trip_trace.json`, `trip_animation.gif`, and `trip_animation_metadata.json` under `experiments/artifacts/live_trip_viz/`.

This notebook is a thin front-end over `sumo_rl.experiments.live_trip_visualization`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from IPython.display import Image as IPyImage, display

try:
    import pandas as pd
except ImportError:
    pd = None


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "sumo_rl").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sumo_rl.experiments.live_trip_visualization import run_best_checkpoint_trip_visualizations

print(f"Repo root: {ROOT}")
print(f"pandas available: {pd is not None}")

In [ ]:
RUN_DIRS = [ROOT / "outputs" / "replace_me_with_hydra_run_dir"]
BEST_INDEX = 0
WIDTH = 1200
FPS = 12
FRAME_COUNT = 180
MAX_RENDER_VEHICLES = 2000
OUTPUT_DIR = None

RUN_DIRS

In [ ]:
RUN_DIRS = [Path(run_dir).expanduser().resolve() for run_dir in RUN_DIRS]
if not RUN_DIRS:
    raise ValueError("RUN_DIRS must contain at least one path.")
for run_dir in RUN_DIRS:
    if not run_dir.exists():
        raise FileNotFoundError(f"RUN_DIR does not exist: {run_dir}")

RESOLVED_OUTPUT_DIR = (
    Path(OUTPUT_DIR).expanduser().resolve()
    if OUTPUT_DIR is not None
    else ROOT / "experiments" / "artifacts" / "live_trip_viz"
)

print("Run dirs:")
for run_dir in RUN_DIRS:
    print(f"- {run_dir}")
print(f"Output root: {RESOLVED_OUTPUT_DIR}")

In [ ]:
results = run_best_checkpoint_trip_visualizations(
    RUN_DIRS,
    RESOLVED_OUTPUT_DIR,
    best_index=int(BEST_INDEX),
    width=int(WIDTH),
    fps=int(FPS),
    frame_count=int(FRAME_COUNT),
    max_render_vehicles=int(MAX_RENDER_VEHICLES),
)

for run_dir, paths in results.items():
    print(f"Run: {run_dir}")
    for label, path in paths.items():
        print(f"  {label}: {path}")

## Preview the GIF

In [ ]:
first_paths = next(iter(results.values()))
display(IPyImage(filename=str(first_paths["animation"])))

## Load saved metadata and trace summary

In [ ]:
metadata = json.loads(Path(first_paths["metadata"]).read_text(encoding="utf-8"))
trace = json.loads(Path(first_paths["trace"]).read_text(encoding="utf-8"))

print(json.dumps(metadata, indent=2)[:4000])
print(f"Total frames in trace: {len(trace.get('frames', []))}")

## Inspect gneJ143 Vehicle Dots

The GIF dots are live TraCI vehicles. Use this section to check which lanes the visible stopped dots actually belong to.

In [ ]:
TARGET_EDGE_PREFIXES = ("10425609#0", "10425609#1")
STOPPED_SPEED_THRESHOLD = 0.1

dot_rows = []
for frame_index, frame in enumerate(trace.get("frames", [])):
    for vehicle in frame.get("vehicles", []):
        lane = str(vehicle.get("lane", ""))
        edge = str(vehicle.get("edge", ""))
        if not (edge.startswith(TARGET_EDGE_PREFIXES) or lane.startswith(TARGET_EDGE_PREFIXES)):
            continue
        speed = float(vehicle.get("speed", 0.0) or 0.0)
        dot_rows.append(
            {
                "frame_index": frame_index,
                "time": float(frame.get("time", 0.0) or 0.0),
                "vehicle_id": str(vehicle.get("id", "")),
                "edge": edge,
                "lane": lane,
                "speed": speed,
                "stopped": speed < STOPPED_SPEED_THRESHOLD,
                "x": float(vehicle.get("x", 0.0) or 0.0),
                "y": float(vehicle.get("y", 0.0) or 0.0),
            }
        )

if pd is None:
    print(f"Matched vehicle-dot rows: {len(dot_rows)}")
    for row in dot_rows[-80:]:
        print(row)
else:
    dots_df = pd.DataFrame(dot_rows)
    display(dots_df.tail(80))
    if not dots_df.empty:
        lane_counts = (
            dots_df.groupby(["time", "lane"], as_index=False)
            .agg(
                vehicles=("vehicle_id", "count"),
                stopped=("stopped", "sum"),
                min_speed=("speed", "min"),
                max_speed=("speed", "max"),
            )
            .sort_values(["time", "lane"])
        )
        display(lane_counts.tail(80))

        segment_counts = (
            dots_df.groupby(["time", "edge"], as_index=False)
            .agg(
                vehicles=("vehicle_id", "count"),
                red_dots=("stopped", "sum"),
                lanes=("lane", lambda values: sorted(set(values))),
            )
            .sort_values(["time", "edge"])
        )
        display(segment_counts.tail(80))
        segment_peaks = segment_counts.sort_values("red_dots", ascending=False).groupby("edge", as_index=False).head(1)
        print("Peak red-dot counts by targeted road segment:")
        display(segment_peaks.sort_values("edge"))
        peak = lane_counts.loc[lane_counts["stopped"].idxmax()]
        print(
            "Peak stopped count on target lanes:",
            int(peak["stopped"]),
            "at time",
            float(peak["time"]),
            "lane",
            peak["lane"],
        )


In [ ]:
if pd is not None and "dots_df" in globals() and not dots_df.empty:
    time_counts = (
        dots_df.groupby("time", as_index=False)
        .agg(
            vehicles=("vehicle_id", "count"),
            stopped=("stopped", "sum"),
            lanes=("lane", lambda values: sorted(set(values))),
        )
        .sort_values("stopped", ascending=False)
    )
    display(time_counts.head(20))

    selected_time = float(time_counts.iloc[0]["time"])
    print(f"Rows at selected peak time: {selected_time}")
    display(dots_df[dots_df["time"] == selected_time].sort_values(["lane", "x", "y"]))
elif pd is None:
    print("Install pandas or inspect the printed rows above for lane membership.")
else:
    print("No vehicles from target edge prefixes were found in the trace.")


In [ ]:
summary_row = {
    "run_dir": metadata.get("run_dir"),
    "checkpoint_rank": metadata.get("checkpoint_rank"),
    "metric_name": metadata.get("metric_name"),
    "metric_value": metadata.get("metric_value"),
    "seed": metadata.get("seed"),
    "trace_frames": metadata.get("trace_frames"),
    "max_live_vehicles": metadata.get("max_live_vehicles"),
    "tls_count": metadata.get("tls_count"),
}

if pd is not None:
    display(pd.DataFrame([summary_row]))
else:
    print(summary_row)

In [ ]:
for path in sorted(RESOLVED_OUTPUT_DIR.rglob("*")):
    print(path.relative_to(RESOLVED_OUTPUT_DIR))